# 04. `UncertainDP` and `ODE_DP`

Two of the more specialised primitives.

**`UncertainDP`** wraps a pair of design problems $(h^L, h^U)$ that bracket a true $h$ which is unknown or non-finitely-representable (Sec. VII). Solving with $h^L$ gives an optimistic Pareto front; solving with $h^U$ gives a pessimistic one. The true minimal resources sit between.

**`ODE_DP`** derives a monotone resource relation from a differential equation: integrate to a final value, or solve for the steady state. Useful when the resource depends on the trajectory of an underlying physical system rather than a closed-form expression.


## Imports

In [1]:
from codesign import (
    AlgebraicDP, NamedProduct, ODE_DP, Reals, UncertainDP, solve,
)

## UncertainDP demo: battery with uncertain specific energy

Old Li-ion cells average 1.6 MJ/kg; newer ones 2.0 MJ/kg. We bracket the unknown true value with the two limits and solve in both modes.


In [2]:
F = NamedProduct({"capacity": Reals(unit="J")})
R = NamedProduct({"mass": Reals(unit="kg")})

pessimistic = AlgebraicDP(
    F=F, R=R,
    equations={"mass": lambda f: f["capacity"] / 1.6e6},
    name="battery_pessimistic",
)
optimistic = AlgebraicDP(
    F=F, R=R,
    equations={"mass": lambda f: f["capacity"] / 2.0e6},
    name="battery_optimistic",
)
uncertain = UncertainDP(F=F, R=R, lower=optimistic, upper=pessimistic, mode="upper")

print("Battery sizing under specific-energy uncertainty (1 kWh capacity):")
for mode in ("lower", "upper"):
    result = solve(uncertain.with_mode(mode), {"capacity": 3.6e6})
    mass = list(result.antichain.points)[0]["mass"]
    label = "optimistic" if mode == "lower" else "pessimistic"
    print(f"   {label:<12} ({mode}): mass = {mass:.3f} kg")
print("\nDesigns that survive the pessimistic case are robust to the uncertainty.")

Battery sizing under specific-energy uncertainty (1 kWh capacity):
   optimistic   (lower): mass = 1.800 kg
   pessimistic  (upper): mass = 2.250 kg

Designs that survive the pessimistic case are robust to the uncertainty.


## ODE_DP demo: steady-state heater

A heated payload loses heat to the environment proportional to its temperature rise (Newton's cooling). At steady state the input power equals the heat-loss coefficient times the temperature delta. The ODE solver finds the steady state by Newton iteration on $\dot x = 0$.


In [3]:
H_LOSS = 0.8  # W/K

heater = ODE_DP(
    F=NamedProduct({"delta_T": Reals(unit="K")}),
    R=NamedProduct({"power": Reals(unit="W")}),
    rhs=lambda x, t, f: H_LOSS * f["delta_T"] - x,
    extract=lambda x: {"power": float(x)},
    mode="steady_state",
    x0_fn=lambda f: 0.0,
    name="heater_ode",
)

print("Power required to hold a steady temperature rise (h_loss = 0.8 W/K):")
for dT in (5.0, 20.0, 50.0):
    result = solve(heater, {"delta_T": dT})
    p = list(result.antichain.points)[0]["power"]
    print(f"   delta_T = {dT:>4.0f} K  ->  P_in = {p:>5.1f} W  "
          f"(=  h_loss * delta_T = {H_LOSS * dT:.1f} W)")

Power required to hold a steady temperature rise (h_loss = 0.8 W/K):
   delta_T =    5 K  ->  P_in =   4.0 W  (=  h_loss * delta_T = 4.0 W)
   delta_T =   20 K  ->  P_in =  16.0 W  (=  h_loss * delta_T = 16.0 W)
   delta_T =   50 K  ->  P_in =  40.0 W  (=  h_loss * delta_T = 40.0 W)


The ODE solver finds the steady state exactly because $\dot x = 0$ has a closed form here. For nonlinear right-hand sides (radiative loss $\propto T^4$, for example) it would still converge as long as the equation is monotone in the resource.
